In [18]:
import numpy as np 
import pandas as pd

chemin_train = '/kaggle/input/datasets/lilianekili/house-princes/house-prices-advanced-regression-techniques/train.csv'
chemin_test = '/kaggle/input/datasets/lilianekili/house-princes/house-prices-advanced-regression-techniques/test.csv'

train = pd.read_csv(chemin_train)
test = pd.read_csv(chemin_test)

print("Taille du jeu d'entraînement :", train.shape)
print("Taille du jeu de test :", test.shape)
train.head()

Taille du jeu d'entraînement : (1460, 81)
Taille du jeu de test : (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [19]:
#  Remplir avec "None" les colonnes où l'absence signifie "Pas de piscine", "Pas de garage"....
colonnes_categorielles = ['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 
                          'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu', 'GarageType', 
                          'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature']
for col in colonnes_categorielles:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

#  Remplir avec 0 les surfaces de garage ou de sous-sol (si pas de garage, surface = 0)
colonnes_numeriques_zeros = ['MasVnrArea', 'GarageYrBlt']
for col in colonnes_numeriques_zeros:
    train[col] = train[col].fillna(0)
    test[col] = test[col].fillna(0)

#  Remplir avec la médiane / le mode pour les quelques trous restants
train['LotFrontage'] = train['LotFrontage'].fillna(train['LotFrontage'].median())
test['LotFrontage'] = test['LotFrontage'].fillna(train['LotFrontage'].median())

train['Electrical'] = train['Electrical'].fillna(train['Electrical'].mode()[0])
test['Electrical'] = test['Electrical'].fillna(test['Electrical'].mode()[0])

In [20]:
for col in test.columns:
    if test[col].isnull().sum() > 0:
        if test[col].dtype == 'object':
            valeur_defaut = train[col].mode()[0]
        else:
            valeur_defaut = train[col].median()
            
        test[col] = test[col].fillna(valeur_defaut)

print("Valeurs manquantes restantes dans le train :", train.isnull().sum().sum())
print("Valeurs manquantes restantes dans le test :", test.isnull().sum().sum())

Valeurs manquantes restantes dans le train : 0
Valeurs manquantes restantes dans le test : 0


In [21]:
y = train['SalePrice'] 
X = train.drop(columns=['SalePrice', 'Id']) 

X_test = test.drop(columns=['Id'])
X = pd.get_dummies(X)
X_test = pd.get_dummies(X_test)
X_test = X_test.reindex(columns=X.columns, fill_value=0)

#Vérification des dimensions
print("Dimensions de X (données d'entraînement) :", X.shape)
print("Dimensions de y (les prix) :", y.shape)
print("Dimensions de X_test (données de test) :", X_test.shape)

Dimensions de X (données d'entraînement) : (1460, 302)
Dimensions de y (les prix) : (1460,)
Dimensions de X_test (données de test) : (1459, 302)


In [22]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

#On sépare nos données d'entraînement en deux : 
#80% pour entraîner le modèle et  20% pour le tester 
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

modele = LinearRegression()
modele.fit(X_train, y_train) 
predictions = modele.predict(X_val)

# 5. On évalue la performance du modèle
# On calcule l'erreur quadratique moyenne (RMSE), c'est l'erreur moyenne en dollars
rmse = np.sqrt(mean_squared_error(y_val, predictions))
print(f"Erreur moyenne du modèle (RMSE) : {rmse:.2f} $")
print(f"Prix moyen des maisons : {y_val.mean():.2f} $")

Erreur moyenne du modèle (RMSE) : 65389.72 $
Prix moyen des maisons : 178839.81 $


In [23]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

modele_rf = RandomForestRegressor(n_estimators=100, random_state=42)

modele_rf.fit(X_train, y_train)

#On fait les prédictions sur les 20% de maisons test
predictions_rf = modele_rf.predict(X_val)

# On calcule la nouvelle erreur
rmse_rf = np.sqrt(mean_squared_error(y_val, predictions_rf))
print(f"Erreur moyenne du Random Forest (RMSE) : {rmse_rf:.2f} $")

Erreur moyenne du Random Forest (RMSE) : 28713.83 $


In [24]:
#On fait les prédictions sur le jeu de test final
predictions_finales = modele_rf.predict(X_test)

#On crée le fichier de soumission avec les IDs et les prix prédits
submission = pd.DataFrame({
    'Id': test['Id'],
    'SalePrice': predictions_finales
})

#On sauvegarde le fichier en CSV
submission.to_csv('submission.csv', index=False)

#On regarde les 5 premières lignes pour vérifier
print("Voici les 5 premières prédictions :")
print(submission.head())

Voici les 5 premières prédictions :
     Id  SalePrice
0  1461  128607.33
1  1462  154015.00
2  1463  177636.84
3  1464  188833.85
4  1465  206305.12
